<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Building a Reflection Agent with LangGraph**


Estimated time needed: **45** minutes


In this guided project, you'll learn to build Reflection agents using LangGraph, a powerful tool for creating self-improving AI systems. Reflection agents represent a significant advancement in AI capabilities - they can evaluate their own outputs, identify weaknesses, and iteratively improve through feedback loops. You'll start by understanding the concept of reflection in AI, then build a complete reflection agent that can generate content, evaluate its quality, and refine outputs through multiple iterations. Working with a LinkedIn post generator as your example, you'll implement a graph-based workflow that mimics human-like reflective thinking, enabling the agent to transform basic content into polished, engaging outputs. By the end of this project, you'll have hands-on experience creating AI systems that can think critically about their own work and continuously improve their performance.


## __Table of Contents__

<ol>
    <li><a href="#Objectives">Objectives</a></li>
    <li>
        <a href="#Setup">Setup</a>
        <ol>
            <li><a href="#Installing-Required-Libraries">Installing Required Libraries</a></li>
        </ol>
    </li>
    <li>
        <a href="#What-is-Reflection?">What is Reflection?</a>
    </li>
    <li>
        <a href="#Workflow-of-Reflection-Agent-in-LangGraph:">Workflow of Reflection Agent in LangGraph</a>
    </li>
    <li>
        <a href="#Building-an-Optimized-LinkedIn-Post-Generator-with-a-Reflection-Agent">Building an Optimized LinkedIn Post Generator with a Reflection Agent</a>
        <ol>
            <li><a href="#Instantiating-the-Language-Model">Instantiating the Language Model</a></li>
            <li><a href="#Generation-Prompt-for-Posts">Generation Prompt for Posts</a></li>
            <li><a href="#Creating-the-Chain-for-LinkedIn-Post-Generation">Creating the Chain for LinkedIn Post Generation</a></li>
            <li><a href="#Reflection-Prompt-for-LinkedIn-Post-Critique">Reflection Prompt for LinkedIn Post Critique</a></li>
            <li><a href="#Creating-the-Reflect-Chain">Creating the Reflect Chain</a></li>
            <li><a href="#Defining-the-Agent-State-for-Reflection-Agent">Defining the Agent State for Reflection Agent</a></li>
            <li><a href="#Defining-the-Generation-and-Reflection--Node">Defining the Generation and Reflection Node</a></li>
            <li><a href="#Why-HumanMessage?">Why HumanMessage?</a></li>
            <li><a href="#Adding-the-Generate-Node-to-the-Graph">Adding the Generate Node to the Graph</a></li>
            <li><a href="#Adding-the-Reflect-Node-to-the-Graph">Adding the Reflect Node to the Graph</a></li>
            <li><a href="#Setting-the-START-Boundary-in-the-Graph">Setting the START Boundary in the Graph</a></li>
            <li><a href="#Adding-a-Routing-Function-for-Decision-Making">Adding a Routing Function for Decision Making</a></li>
            <li><a href="#Compiling-the-Workflow">Compiling the Workflow</a></li>
            <li><a href="#Defining-Inputs-for-the-Workflow">Defining Inputs for the Workflow</a></li>
            <li><a href="#Executing-the-Workflow">Executing the Workflow</a></li>
            <li><a href="#Plotting-the-Graph">Plotting the Graph</a></li>
        </ol>
    </li>
</ol>


## Objectives

After completing this lab you will be able to:

- Build reflection-enabled AI agents using LangGraph's graph-based workflow structure
- Implement a multi-step process that generates, evaluates, and refines AI-produced content
- Design conversational workflows with message state management for improved context retention
- Create conditional routing logic to control agent behavior and iteration processes
- Apply reflection techniques to enhance the quality and accuracy of AI-generated content
- Develop self-improving systems that can identify and address their own limitations


----


## Setup


For this lab, we will be using the following libraries:

*   [`langgraph`](https://pypi.org/project/langgraph/) for building stateful graph workflows.
*   [`langchain[openai]`](https://pypi.org/project/langchain/) for LangChain's current OpenAI chat model integration.
*   [`python-dotenv`](https://pypi.org/project/python-dotenv/) for loading `OPENAI_API_KEY` and optional model settings from a local `.env` file.


### Installing Required Libraries


In [1]:
# %pip install -q -U "langgraph" "langchain[openai]" "python-dotenv"

In [4]:
import os
from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import END, START, MessagesState, StateGraph

load_dotenv()

True

# API Configuration

This lab uses OpenAI chat models through LangChain. Store your API key in a local `.env` file instead of hard-coding it in the notebook.

Example `.env` file:

```text
OPENAI_API_KEY=your_openai_api_key_here
OPENAI_MODEL=gpt-5-nano
```

`OPENAI_MODEL` is optional. If it is not set, the notebook uses `gpt-5-nano`.


In [5]:
# The model setup cell below calls load_dotenv(), checks OPENAI_API_KEY,
# and creates the shared OpenAI chat model.
#
# Keep secrets in .env. Do not paste API keys directly into notebook cells.


# What is Reflection?
Reflection is a prompting strategy aimed at enhancing the quality and accuracy of outputs generated by AI agents. It involves getting the agent to **pause, review, and critique** its own outputs before finalizing them. This iterative process helps in reducing errors and improving performance over time.

For example, when an AI model generates code, it typically outputs the result instantly. However, just like human programmers, code needs to be tested and refined. Reflection ensures that the AI agent **evaluates the generated code, identifies potential errors, and iterates** to fix them. This mimics how developers write, test, debug, and optimize their work, resulting in more reliable outputs.

A simple analogy is comparing it to having two systems:
- **System 1** – Reactive and instinctive (quick initial responses).
- **System 2** – Reflective and deliberate (carefully reviewing and refining outputs).

Reflection agents encourage AI to function more like **System 2**, iterating over their work until the desired quality is achieved.


## Workflow of Reflection Agent in LangGraph:

1. **Generation Node: Generate Initial Output**
   - The first step in the process is the **generation node**, which quickly produces an initial output based on the given prompt. This stage is all about generating a first draft without focusing too much on perfection. It acts like an instinctive response, providing a rough version of the output. For example, if the task is to write a LinkedIn post, the generation node would come up with a basic idea. This draft is then passed to the next step for evaluation.
     
<br>

2. **Evaluation Node: Evaluate Output for Quality**
   - After the initial output is generated, the **evaluation node** assesses its quality. This step is about checking if the output is good enough or if it needs improvement. The evaluation focuses on key aspects like whether the message is clear, engaging, and relevant. For instance, in the case of a LinkedIn post, the evaluation might decide if the post feels authentic, aligns with professional tone, or misses important context. If the output is deemed acceptable, it moves forward to the final step.

<br>

3. **Reflection Node: Critique and Refine**
   - If the evaluation node determines that the output needs improvement, the **reflection node** steps in to refine the content. This step is more thoughtful and deliberate, where the system reflects on the output and looks for ways to improve it. The reflection node critiques the draft, suggests changes, and revises the content to make it more polished. It could involve enhancing tone, adding clarity, highlighting achievements, or making the post more engaging. The system keeps refining the output through this process until it reaches the desired quality level.
     
<br>

4. **Final Output: Present Refined Result**
   - Once the reflection node has done its job, the final output is produced. This is the result of the reflection and evaluation processes, where the initial draft has been refined and improved. The agent now presents the final version of the content — a polished, high-quality LinkedIn post ready for publishing. After this step, the process concludes, and the final response is delivered to the user.


---

**Example (LinkedIn Post Generation):**  
Imagine asking an AI to write a LinkedIn post announcing a job promotion:  

**Prompt:**  
"Write a LinkedIn post announcing my promotion to Engineering Manager."  

**AI’s Initial Output (System 1):**  
*"Excited to share that I’ve been promoted to Engineering Manager!"*  

**Reflection Step:**  
The AI reviews the post and asks:  
*"Does this post highlight leadership growth or express gratitude?"*  

**Refined Output (System 2):**  
*"I'm thrilled to share that I've been promoted to Engineering Manager at [Company]! Grateful for the mentorship, team collaboration, and opportunities that led to this moment. Looking forward to leading new initiatives and continuing to grow with this incredible team. #Leadership #CareerGrowth #EngineeringManager"*

---


### **Building an Optimized LinkedIn Post Generator with a Reflection Agent**

In this process, we aim to enhance the quality of AI-generated posts using a Reflection Agent. The idea is to allow the AI to generate a post and then critique its own output, refining the content iteratively based on feedback. This approach helps improve the engagement, relevance, and tone, ensuring a better final result.

We will be building a system that includes a generation phase where the post is generated, followed by a reflection phase where the AI reviews and refines the output. This cycle ensures that the AI produces higher-quality content in the end.

The following diagram illustrates the workflow of this system, showing the interaction between the generation and reflection nodes.


<div style="text-align: center;">
  <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/Cbuc3z8N1_Ew2ESw199Slw/Workflow.png" alt="Workflow" style="width: 40%; height: 500px;">
</div>


### **Instantiating the Language Model**

This step loads environment variables from `.env` and initializes an OpenAI chat model with LangChain's current `init_chat_model` interface.


In [6]:
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY is not set. Create a .env file with OPENAI_API_KEY=your-key."
    )

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")
llm = init_chat_model(OPENAI_MODEL, model_provider="openai")
llm


ChatOpenAI(output_version=None, profile={'name': 'GPT-5 Nano', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x12fb49c10>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x12f953740>, root_client=<openai.OpenAI object at 0x12eece090>, root_async_client=<openai.AsyncOpenAI object at 0x12fbc1610>, model_name='gpt-5-nano', model_kwargs={}, openai_api_key=SecretStr('***

### **Generation Prompt for Posts**

In this section, we are creating a generation prompt for generating LinkedIn posts. The assistant is tasked with crafting high-quality post based on the user's input. Additionally, if the user provides feedback or critique, the assistant revises the post content accordingly.


We are using **`ChatPromptTemplate`** from LangChain to structure the prompt. The prompt has two main parts:

1. **System Message**:  
   This provides instructions to the assistant about its role and task.  
   Here, the assistant is framed as a **professional LinkedIn content assistant** who is expected to generate the best possible LinkedIn post based on the user's input.  
   It also specifies that if the user provides feedback or critique, the assistant should revise the post to improve clarity, tone, or engagement.

2. **MessagesPlaceholder**:  
   This is used to inject the actual content or message that the post will be based on.  
   The placeholder will be populated with the user’s request at runtime.


In [7]:
generation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a professional LinkedIn content assistant tasked with crafting engaging, insightful, and well-structured LinkedIn posts."
            " Generate the best LinkedIn post possible for the user's request."
            " If the user provides feedback or critique, respond with a refined version of your previous attempts, improving clarity, tone, or engagement as needed.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

### **Creating the Chain for LinkedIn Post Generation**

In this step, we are combining the **`generation_prompt`** with a language model (LLM) to form a complete chain that will allow the system to generate LinkedIn posts based on user input.

The **`generate_chain`** links the **`generation_prompt`** with the **LLM** (Large Language Model), enabling the system to generate a professional LinkedIn post after processing the user's input through the prompt.

- **`generation_prompt`**: This is the template that guides the model on how to generate the LinkedIn post, including the system message and the placeholder for the user's input.
- **`llm`**: This is the language model that will take the prompt and produce the post based on the input provided.

By using the pipe operator (`|`), we are chaining these components together so that the prompt flows seamlessly into the language model and the model generates the final LinkedIn post.


In [8]:
generate_chain = generation_prompt | llm

<br>

### **Reflection Prompt for LinkedIn Post Critique**

In this step, we define the **`reflection_prompt`**, which is a template used for generating critiques and recommendations to improve a user's LinkedIn post. This prompt guides the model to assess the quality of a LinkedIn post and provide structured, actionable feedback.

- **`reflection_prompt`**: The system message here instructs the model to act as a professional LinkedIn content strategist. The model will evaluate the post based on factors like tone, structure, clarity, engagement potential, formatting, and relevance. It then generates feedback to help improve the post’s overall effectiveness and alignment with LinkedIn best practices.


In [9]:
reflection_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a professional LinkedIn content strategist and thought leadership expert. Your task is to critically evaluate the given LinkedIn post and provide a comprehensive critique. Follow these guidelines:

        1. Assess the post’s overall quality, professionalism, and alignment with LinkedIn best practices.
        2. Evaluate the structure, tone, clarity, and readability of the post.
        3. Analyze the post’s potential for engagement (likes, comments, shares) and its effectiveness in building professional credibility.
        4. Consider the post’s relevance to the author’s industry, audience, or current trends.
        5. Examine the use of formatting (e.g., line breaks, bullet points), hashtags, mentions, and media (if any).
        6. Evaluate the effectiveness of any call-to-action or takeaway.

        Provide a detailed critique that includes:
        - A brief explanation of the post’s strengths and weaknesses.
        - Specific areas that could be improved.
        - Actionable suggestions for enhancing clarity, engagement, and professionalism.

        Your critique will be used to improve the post in the next revision step, so ensure your feedback is thoughtful, constructive, and practical.
        """
    ),
    MessagesPlaceholder(variable_name="messages")
])

### **Creating the Reflect Chain**
The **`reflect_chain`** is created by combining the **`reflection_prompt`** with the language model (LLM). This chain allows the model to evaluate and provide feedback on the generated post.


In [10]:
reflect_chain = reflection_prompt | llm

### **Defining the Agent State for Reflection Agent**

Reflection agents need to keep the full conversation history:

- the original user request
- generated drafts
- critique messages
- revised drafts

Current LangGraph examples use `MessagesState` for this pattern. It is a built-in state schema with a `messages` key and message-appending behavior, so each node can return only the new messages it produced.

A manual equivalent would use `add_messages`:

```python
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
```

For this lab, we use `MessagesState` directly.


---
#### **LangGraph's `MessagesState`**

`MessagesState` replaces the older `MessageGraph` style for modern LangGraph examples.

**Features of `MessagesState`:**
1. Stores the conversation under `state["messages"]`.
2. Appends returned messages instead of replacing history.
3. Works naturally with chat models and LangChain message objects.
4. Can be extended with extra state fields later if needed.

---


#### **Initializing `StateGraph(MessagesState)`**

We initialize a regular `StateGraph` with `MessagesState`. This keeps the API consistent with other LangGraph workflows and uses explicit `START` and `END` boundaries.


In [11]:
graph = StateGraph(MessagesState)


Behind the Scenes:

- As the user interacts with the agent, **`HumanMessage`** is added to the state.  
- The AI generates a response (**`AIMessage`**), which gets appended to the list.  
- If the output needs improvement, a **`SystemMessage`** can trigger a reflection phase to refine the response.


### **Defining the Generation and Reflection Nodes**

The `generation_node` creates or revises the LinkedIn post.

- **Input**: `MessagesState`, accessed through `state["messages"]`.
- **Output**: a partial state update: `{"messages": [AIMessage(...)]}`.
- LangGraph appends the returned message to the existing message history.


In [12]:
def generation_node(state: MessagesState) -> dict:
    generated_post = generate_chain.invoke({"messages": state["messages"]})
    return {"messages": [AIMessage(content=generated_post.content)]}


The `reflection_node` critiques the latest generated post. It returns the critique as a `HumanMessage` so the next generation step treats it as feedback from a reviewer.


In [13]:
def reflection_node(state: MessagesState) -> dict:
    critique = reflect_chain.invoke({"messages": state["messages"]})
    return {"messages": [HumanMessage(content=critique.content)]}


<br>

### **Why `HumanMessage`?**

The output is wrapped in a `HumanMessage` because the reflection process is a form of feedback or critique given to the **generation agent**, and the feedback is intended to be treated as if it is coming from the user. This is important for the iterative process where the AI generates content and then receives human-like feedback to improve the output. In the context of this workflow, we treat the feedback as if a human is guiding the reflection agent to enhance its output.

- **HumanMessage** here is not used to represent user input directly but rather to provide feedback (as if from a human perspective). This feedback is passed back into the system, enabling the generation agent to revise its content. 
- It effectively gives the reflection node the authority to "speak" to the generation node, but in the context of providing critique and recommendations for refinement.


### **Adding the Generate Node to the Graph**

Now we add the generation node to the graph using the `add_node` function. This function takes two parameters:  

1. **Name**: A unique identifier for the node, in this case, `"generate"`.  
2. **Function**: The function to be executed when this node is triggered, here `generation_node`.  


In [14]:
graph.add_node("generate", generation_node)

We can summarize the process with the image: the generation prompt chains to the LLM using the generate chain. The generate node invokes this chain, storing the LLM's output in a sequence (list) of messages. The Graph object is created in red. Finally, add_node represents this as a blue node in the graph.




![image](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/eWlf-wNPOye4o_B0Ax-3mg/Generate.png)


<br>

### **Adding the Reflect Node to the Graph**

We now add the reflection node to the graph using the `add_node` function. This function takes two parameters:  

1. **Name**: A unique identifier for the node, in this case, `"reflect"`.  
2. **Function**: The function to be executed when this node is triggered, here `reflection_node`.  

This step integrates the `"reflect"` node into the graph, linking it to the `reflection_node` function. This node is responsible for providing feedback and suggestions for improving the generated content, enabling the reflection phase of the agent.


In [15]:
graph.add_node("reflect", reflection_node)

Similarly, we represent the creation of the reflect node with an image where the reflect node `reflect_chain.invoke({"messages": messages})` maps its input to the sequence of inputs to the LLM and returns new HumanMessages in a list format. The Graph object is represented in Red. The add_node method adds this to the graph object represented as a green node.
![reflect.png](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/-nFI6IESRnfmlF2tgeWr3g/reflect.png)


The method `graph.add_edge("reflect", "generate")` creates the loop. After the reflection node adds critique, control returns to the generation node so the model can revise the post.


In [16]:
graph.add_edge("reflect", "generate")


!![Screenshot 2025-02-10 at 4.24.25 PM.png](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/Z5jtZS3hNxh4jJROKeUa5Q/Screenshot%202025-02-10%20at%204-24-25%E2%80%AFPM.png)


<br>

### **Setting the START Boundary in the Graph**

Modern LangGraph examples use an explicit edge from `START` to the first real node. Here, execution begins at `generate`.


In [17]:
graph.add_edge(START, "generate")


We can think of `START` as the graph's virtual entry boundary. It is not a node that we implement ourselves.


<br>

### **Adding a Routing Function for Decision Making**

The routing function decides whether the workflow should continue to reflection or stop at `END`.

1. **Predefined logic**: check the number of messages in `state["messages"]`. If it exceeds the limit, end the workflow. Otherwise, continue to `reflect`.
2. **LLM-based logic**: a later version could ask a model whether more reflection is useful.

For this lab, we use predefined logic so the loop is deterministic and easy to inspect.


In [18]:
def should_continue(state: MessagesState) -> Literal["reflect", END]:
    if len(state["messages"]) > 6:
        return END
    return "reflect"


Since the reflection loop should not run forever, we use `add_conditional_edges`. After each `generate` step, LangGraph calls `should_continue(state)`:

- If it returns `"reflect"`, the graph moves to the reflection node.
- If it returns `END`, the workflow stops.


In [19]:
graph.add_conditional_edges("generate", should_continue)


![add_cond_node.png(1)](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/cVhRRFclksKny2JQUAcTiA/add-cond-node.png)


### **Compiling the Workflow**

Now we compile the workflow using `graph.compile()`. This step ensures that all nodes, edges, and conditional logic defined in the graph are connected and ready for execution.  


In [20]:
workflow = graph.compile()


### **Defining Inputs for the Workflow**

In this example, we define the initial user input as a `HumanMessage`. This message contains a request to improve a post related to LangChain's Tool Calling feature. The content provides the context for the workflow, which will process it and generate a refined output.


In [21]:
inputs = {
    "messages": [
        HumanMessage(
            content="Write a LinkedIn post about getting a software developer job under 160 characters"
        )
    ]
}


### **Executing the Workflow**

Once the input has been defined, the workflow is executed using `workflow.invoke(inputs)`. The compiled graph starts from `START`, runs the `generate` node, and follows the conditional loop until it reaches `END`.


In [22]:
response = workflow.invoke(inputs)


Now, let's see the first generated LinkedIn without any critique.


In [23]:
response["messages"][1].content


'Just landed a Software Developer role! Grateful for mentors, peers, and the journey ahead. Excited to build meaningful software.'

Now, let's see the first critique for this generated post.


In [24]:
response["messages"][2].content


'Here’s a structured critique of the posted content and practical improvements you can apply.\n\nOverall read\n- Strengths: Positive, concise, celebratory. Fits the under-160-character constraint. Clear gratitude and forward-looking sentiment.\n- Weaknesses: Lacks specificity, engagement hooks, and discoverability cues (hashtags, mentions, CTA). Reads a bit generic and could miss opportunities to build credibility or connections.\n\nStrengths in detail\n- Professional tone: Appropriate for LinkedIn; avoids slang and keeps it career-focused.\n- Brevity: Fits platform norms for quick, scannable updates.\n- Positive framing: Expresses gratitude and optimism, which resonates with networks.\n\nKey areas to improve\n- Specificity: Mentioning the company (if publicly shareable) or the tech stack adds credibility and relevance.\n- Engagement/CTA: Include a prompt to connect, discuss opportunities, or offer advice.\n- Discoverability: Hashtags and mentions help the post reach a broader audience

---
As we can see, this is the first critique of our generated post. It highlights both strengths and areas for improvement, offering specific suggestions to enhance engagement, relevance, and impact. Next, we’ll refine the post based on this feedback and generate an improved version.


Now, let's see the final generated response after multiple iterations incorporating the feedback.


In [25]:
response["messages"][-1].content


'Nice work refining. Here are ready-to-use final variants under 160 chars. They keep two lines, a clear CTA, and two hashtags.\n\nOption A — with company and stack\nJust landed a Software Developer role at [Company]!\nGrateful for mentors. Building with React & Node. DM for opportunities. #SoftwareDevelopment #OpenToWork\n\nOption B — strong CTA (no company)\nJust landed a Software Developer role!\nGrateful for mentors. Ready to ship with React & Node. DM to connect. #SoftwareDevelopment #TechCareers\n\nOption C — broader stack + impact\nJust landed as Software Developer!\nGrateful for mentors. Ready to build with React, TS & Node. DM to chat opportunities. #SoftwareDevelopment #OpenToWork\n\nOption D — concise with portfolio\nJust landed as Software Developer.\nGrateful for mentors. DM for React opportunities. Portfolio: [link] #SoftwareDevelopment #OpenToWork\n\nTips\n- Use 2 lines for mobile readability; keep CTA clear.\n- Keep hashtags to 2–3 total; swap in your most relevant ones.

This table tracks the state transitions in a reflection agent's workflow. Each row represents one appended message. After several generate/reflect cycles, the conditional edge routes to `END`.


| Iteration | Message Type | State Change | Active Node | Next Action |
|-----------|--------------|--------------|-------------|-------------|
| 1 | Human | Initial request | Input | Generate |
| 1 | AI | First generated post | Generate | Reflect |
| 1 | Human | Critique feedback | Reflect | Generate |
| 2 | AI | Revised post | Generate | Reflect |
| 2 | Human | More feedback | Reflect | Generate |
| 3 | AI | Final generated post | Generate | END |


---
As we can see, the final LinkedIn post incorporates feedback by adding a compelling statistic, emphasizing urgency, and suggesting a relevant election topic. It also encourages engagement with a question and includes a visual element, making it more engaging and shareable.


#### **Plotting the Graph**

In this step, we visualize the workflow graph as Mermaid. The `workflow.get_graph().draw_mermaid()` method returns Mermaid syntax that can be rendered directly in the notebook.


In [26]:
from IPython.display import Markdown, display

mermaid = workflow.get_graph().draw_mermaid()
display(Markdown(f"```mermaid\n{mermaid}\n```"))


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	generate(generate)
	reflect(reflect)
	__end__([<p>__end__</p>]):::last
	__start__ --> generate;
	generate -.-> __end__;
	generate -.-> reflect;
	reflect --> generate;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

## Authors


[Kunal Makwana](https://author.skills.network/instructors/kunal_makwana) is a Data Scientist at IBM and is currently pursuing his Master's in Computer Science at Dalhousie University.


### Other Contributors


[Joseph Santarcangelo](https://author.skills.network/instructors/joseph_santarcangelo)


[Karan Goswami](https://author.skills.network/instructors/karan_goswami) is a Data Scientist at IBM and is currently pursuing his Masters in Engineering at McMaster University.


Copyright © 2025 IBM Corporation. All rights reserved.
